# 🧬 01: Kök Hücre scRNA-seq Analizi ve Waddington Epigenetik Manzarası

Bu notebookta:
1. Sentetik tek hücre (scRNA-seq) verisi üretimi ve kalite kontrolü
2. Kütüphane normalizasyonu ve log1p dönüşümü
3. En değişken genlerin (HVG) seçimi ve PCA projeksiyonu
4. Pluripotensi genleri (OCT4, SOX2, NANOG) analizi
5. Waddington epigenetik manzarası ve Langevin reprogramlama simülasyonu incelenir.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.scrna_pipeline import generate_synthetic_stem_cell_data, ScRNAPipeline
from src.waddington import WaddingtonLandscape, simulate_reprogramming
from src.stemness_index import calculate_stemness_index

print("Kütüphaneler başarıyla yüklendi!")

In [ ]:
# 1. Tek Hücre Verisinin Üretilmesi ve İncelenmesi
counts_df, meta_df = generate_synthetic_stem_cell_data(n_cells=400, seed=42)
print(f"Hücre Sayısı: {counts_df.shape[0]}, Gen Sayısı: {counts_df.shape[1]}")
print(meta_df["cell_type"].value_counts())

In [ ]:
# 2. Pipeline Ön İşleme ve PCA Analizi
pipe = ScRNAPipeline()
log1p_df = pipe.preprocess(counts_df)
hvgs = pipe.find_highly_variable_genes(top_n=25)
pca_coords = pipe.run_pca(n_components=2)
clusters = pipe.cluster_cells(n_clusters=5)
pseudotime = pipe.compute_pseudotime()
stemness = calculate_stemness_index(log1p_df)

plt.figure(figsize=(8, 6))
plt.scatter(pca_coords["PC_1"], pca_coords["PC_2"], c=stemness, cmap="viridis", alpha=0.8)
plt.colorbar(label="Kök Hücrelilik İndeksi (mRNAsI)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("scRNA-seq PCA Projeksiyonu ve Stemness Skoru")
plt.show()

In [ ]:
# 3. Waddington Epigenetik Manzarası ve Reprogramlama
sim = simulate_reprogramming(start_pos=(1.4, 0.7), yamanaka_strength=0.85, steps=150, seed=42)
traj = sim["trajectory"]

plt.figure(figsize=(8, 5))
plt.plot(traj[:, 0], traj[:, 1], color="crimson", lw=2.5, marker="o", markersize=3, label="Hücre Kader Yörüngesi")
plt.scatter([traj[0, 0]], [traj[0, 1]], color="blue", s=100, label="Başlangıç: Somatik")
plt.scatter([traj[-1, 0]], [traj[-1, 1]], color="green", s=100, label=f"Bitiş: {sim['final_state']}")
plt.xlabel("Farklılaşma Ekseni")
plt.ylabel("Lineage Kolu")
plt.title("Yamanaka Faktörleri ile iPSC Reprogramlama Simülasyonu")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()